In [1]:
import pandas as pd
import re
import io


In [2]:
with open('/Users/pranavaditya/Desktop/Data-Engineering-Journey/First_Pipeline/data-ingestion/support-logs/day-wise-logs-data/support_logs_2025-07-01.log') as f:
    content = f.read()
len(content)

32938

In [ ]:
lines = content.split('---')
lines = [entry.strip() for entry in content.split('---') if entry.strip()]

'2025-07-01 14:10:00 [INFO] careplus.support.GenericService - TicketID=TCK0701029 SessionID=sess_TCK0701029\nIP=178.77.232.8 | ResponseTime=214ms | CPU=33.47% | EventType=generic_event | Error=false\nUserAgent="curl/7.68.0"\nMessage=" event for TCK0701029"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None'

In [16]:
import re


fields = {
    "timestamp": r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})",
    "log_level": r"\[(.*?)\]",
    "service": r"\] ([\w\.]+) -",
    "ticket_id": r"TicketID=(\S+)",
    "session_id": r"SessionID=(\S+)",
    "ip": r"IP=([\d.]+)",
    "response_time": r"ResponseTime=(\d+)ms",
    "cpu": r"CPU=([\d.]+)%",
    "event_type": r"EventType=(\w+)",
    "error": r"Error=(\w+)",
    "user_agent": r'UserAgent="(.*?)"',
    "message": r'Message="(.*?)"',
    "debug": r'Debug="(.*?)"',
    "trace_id": r"TraceID=(\S+)"
}

all_logs = []

for log in lines:
    extracted = {}
    for key, pattern in fields.items():
        match = re.search(pattern, log, re.DOTALL)
        extracted[key] = match.group(1) if match else None
    all_logs.append(extracted)

print(all_logs)

[{'timestamp': '2025-07-01 00:21:00', 'log_level': 'INF0', 'service': 'careplus.support.GenericService', 'ticket_id': 'TCK0701000', 'session_id': 'sess_TCK0701000', 'ip': '60.130.155.7', 'response_time': '1269', 'cpu': '27.64', 'event_type': 'generic_event', 'error': 'false', 'user_agent': 'PostmanRuntime/7.32.2', 'message': ' event for TCK0701000', 'debug': 'ℹ️ Logged for monitoring', 'trace_id': 'None'}, {'timestamp': '2025-07-01 00:41:00', 'log_level': 'INFO', 'service': 'careplus.support.GenericService', 'ticket_id': 'TCK0701000', 'session_id': 'sess_TCK0701000', 'ip': '58.36.189.27', 'response_time': '1505', 'cpu': '57.24', 'event_type': 'generic_event', 'error': 'false', 'user_agent': 'Mobile-Safari/537.36', 'message': ' event for TCK0701000', 'debug': 'ℹ️ Logged for monitoring', 'trace_id': 'None'}, {'timestamp': '2025-07-01 01:44:00', 'log_level': 'DEBUG', 'service': 'careplus.support.GenericService', 'ticket_id': 'TCK0701001', 'session_id': 'sess_TCK0701001', 'ip': '181.18.12.

In [18]:
df = pd.DataFrame(all_logs)
df.head()

,timestamp,log_level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring,None
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,Mobile-Safari/537.36,event for TCK0701000,ℹ️ Logged for monitoring,None
2,2025-07-01 01:44:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,181.18.12.170,586,78.43,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring,None
3,2025-07-01 01:49:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,163.214.94.42,878,63.61,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring,None
4,2025-07-01 01:50:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,155.68.207.12,1614,87.85,generic_event,false,Python-urllib/3.9,event for TCK0701000,ℹ️ Logged for monitoring,None


In [19]:
df.describe()

,timestamp,log_level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id
count,105,105,105,105,105,105,98,105,105,105,105,105,105,105
unique,90,6,1,30,30,96,85,96,1,1,5,30,1,1
top,2025-07-01 03:30:00,INFO,careplus.support.GenericService,TCK0701028,sess_TCK0701028,36.64.191.144,1253,81.83,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701028,ℹ️ Logged for monitoring,None
freq,2,40,105,7,7,2,3,2,105,105,27,7,105,105


In [22]:
df = df.drop('trace_id',axis=1)

In [23]:
df

,timestamp,log_level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,Mobile-Safari/537.36,event for TCK0701000,ℹ️ Logged for monitoring
2,2025-07-01 01:44:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,181.18.12.170,586,78.43,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring
3,2025-07-01 01:49:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,163.214.94.42,878,63.61,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring
4,2025-07-01 01:50:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,155.68.207.12,1614,87.85,generic_event,false,Python-urllib/3.9,event for TCK0701000,ℹ️ Logged for monitoring
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,2025-07-01 13:01:00,INFO,careplus.support.GenericService,TCK0701027,sess_TCK0701027,198.203.219.107,816,75.64,generic_event,false,Mobile-Safari/537.36,event for TCK0701027,ℹ️ Logged for monitoring
101,2025-07-01 13:03:00,INFO,careplus.support.GenericService,TCK0701025,sess_TCK0701025,140.58.166.39,664,14.77,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701025,ℹ️ Logged for monitoring
102,2025-07-01 13:13:00,INF0,careplus.support.GenericService,TCK0701028,sess_TCK0701028,162.176.236.74,585,88.27,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701028,ℹ️ Logged for monitoring
103,2025-07-01 13:46:00,DEBG,careplus.support.GenericService,TCK0701029,sess_TCK0701029,234.150.37.60,1064,30.43,generic_event,false,curl/7.68.0,event for TCK0701029,ℹ️ Logged for monitoring


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   timestamp      105 non-null    str  
 1   log_level      105 non-null    str  
 2   service        105 non-null    str  
 3   ticket_id      105 non-null    str  
 4   session_id     105 non-null    str  
 5   ip             105 non-null    str  
 6   response_time  98 non-null     str  
 7   cpu            105 non-null    str  
 8   event_type     105 non-null    str  
 9   error          105 non-null    str  
 10  user_agent     105 non-null    str  
 11  message        105 non-null    str  
 12  debug          105 non-null    str  
dtypes: str(13)
memory usage: 30.0 KB


In [25]:
df = df.astype({"cpu": "float"})
df["response_time"] = pd.to_numeric(df["response_time"], errors="coerce").astype("Int64")
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      105 non-null    datetime64[us]
 1   log_level      105 non-null    str           
 2   service        105 non-null    str           
 3   ticket_id      105 non-null    str           
 4   session_id     105 non-null    str           
 5   ip             105 non-null    str           
 6   response_time  98 non-null     Int64         
 7   cpu            105 non-null    float64       
 8   event_type     105 non-null    str           
 9   error          105 non-null    str           
 10  user_agent     105 non-null    str           
 11  message        105 non-null    str           
 12  debug          105 non-null    str           
dtypes: Int64(1), datetime64[us](1), float64(1), str(10)
memory usage: 27.3 KB


In [27]:
df.describe()

,timestamp,response_time,cpu
count,105,98.0,105.000000
mean,2025-07-01 08:21:31.428571,1006.5,54.918190
min,2025-07-01 00:21:00,126.0,10.140000
25%,2025-07-01 05:38:00,653.25,34.420000
50%,2025-07-01 09:13:00,1089.0,60.140000
75%,2025-07-01 11:12:00,1327.75,73.700000
max,2025-07-01 14:10:00,1792.0,89.970000
std,NaN,447.808944,22.513155


In [28]:
df['log_level'].value_counts()

log_level
INFO       40
DEBUG      36
INF0       13
DEBG       10
warnING     3
WARNING     3
Name: count, dtype: int64

In [29]:
fixes = {'INF0':'INFO','warnING':'WARNING','DEBG':'DEBUG'}
df['log_level'] = df['log_level'].replace(fixes)
df.log_level.value_counts()

log_level
INFO       53
DEBUG      46
WARNING     6
Name: count, dtype: int64

In [30]:
df[df.duplicated()]

,timestamp,log_level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
10,2025-07-01 03:30:00,DEBUG,careplus.support.GenericService,TCK0701002,sess_TCK0701002,36.64.191.144,1223,81.83,generic_event,false,curl/7.68.0,event for TCK0701002,ℹ️ Logged for monitoring
33,2025-07-01 06:26:00,INFO,careplus.support.GenericService,TCK0701009,sess_TCK0701009,30.228.28.191,1253,32.54,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701009,ℹ️ Logged for monitoring
38,2025-07-01 06:43:00,DEBUG,careplus.support.GenericService,TCK0701008,sess_TCK0701008,214.140.181.78,1372,63.43,generic_event,false,Mobile-Safari/537.36,event for TCK0701008,ℹ️ Logged for monitoring
44,2025-07-01 07:57:00,INFO,careplus.support.GenericService,TCK0701018,sess_TCK0701018,167.18.200.246,1454,85.97,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701018,ℹ️ Logged for monitoring
57,2025-07-01 09:32:00,DEBUG,careplus.support.GenericService,TCK0701013,sess_TCK0701013,114.173.55.131,1089,32.70,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701013,ℹ️ Logged for monitoring
59,2025-07-01 09:36:00,DEBUG,careplus.support.GenericService,TCK0701020,sess_TCK0701020,146.157.172.98,808,78.08,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701020,ℹ️ Logged for monitoring
69,2025-07-01 10:17:00,INFO,careplus.support.GenericService,TCK0701019,sess_TCK0701019,163.229.213.118,1568,24.09,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701019,ℹ️ Logged for monitoring
77,2025-07-01 10:45:00,INFO,careplus.support.GenericService,TCK0701023,sess_TCK0701023,30.211.17.103,1127,22.14,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701023,ℹ️ Logged for monitoring
92,2025-07-01 12:25:00,DEBUG,careplus.support.GenericService,TCK0701028,sess_TCK0701028,138.124.89.86,1250,46.43,generic_event,false,Mobile-Safari/537.36,event for TCK0701028,ℹ️ Logged for monitoring


In [38]:
df[df.duplicated()]

,timestamp,log_level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
10,2025-07-01 03:30:00,DEBUG,careplus.support.GenericService,TCK0701002,sess_TCK0701002,36.64.191.144,1223,81.83,generic_event,false,curl/7.68.0,event for TCK0701002,ℹ️ Logged for monitoring
33,2025-07-01 06:26:00,INFO,careplus.support.GenericService,TCK0701009,sess_TCK0701009,30.228.28.191,1253,32.54,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701009,ℹ️ Logged for monitoring
38,2025-07-01 06:43:00,DEBUG,careplus.support.GenericService,TCK0701008,sess_TCK0701008,214.140.181.78,1372,63.43,generic_event,false,Mobile-Safari/537.36,event for TCK0701008,ℹ️ Logged for monitoring
44,2025-07-01 07:57:00,INFO,careplus.support.GenericService,TCK0701018,sess_TCK0701018,167.18.200.246,1454,85.97,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701018,ℹ️ Logged for monitoring
57,2025-07-01 09:32:00,DEBUG,careplus.support.GenericService,TCK0701013,sess_TCK0701013,114.173.55.131,1089,32.70,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701013,ℹ️ Logged for monitoring
59,2025-07-01 09:36:00,DEBUG,careplus.support.GenericService,TCK0701020,sess_TCK0701020,146.157.172.98,808,78.08,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701020,ℹ️ Logged for monitoring
69,2025-07-01 10:17:00,INFO,careplus.support.GenericService,TCK0701019,sess_TCK0701019,163.229.213.118,1568,24.09,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701019,ℹ️ Logged for monitoring
77,2025-07-01 10:45:00,INFO,careplus.support.GenericService,TCK0701023,sess_TCK0701023,30.211.17.103,1127,22.14,generic_event,false,Mozilla/5.0 (Windows NT 10.0),event for TCK0701023,ℹ️ Logged for monitoring
92,2025-07-01 12:25:00,DEBUG,careplus.support.GenericService,TCK0701028,sess_TCK0701028,138.124.89.86,1250,46.43,generic_event,false,Mobile-Safari/537.36,event for TCK0701028,ℹ️ Logged for monitoring


In [39]:
df = df.drop_duplicates()
df[df.duplicated()]

,timestamp,log_level,service,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug


In [40]:
df.shape

(96, 13)